# 

In [17]:
import pandas as pd

for f in ["eaurob-2024\\UDI_PLV_2024.txt", "eaurob-2024\\UDI_RES_2024.txt"]:
    df = pd.read_csv(f, sep=",", nrows=1000, low_memory=False)
    print(f, df.shape)
    print(df.columns.tolist())
    print(df.head(3).T)
    print("-" * 40)


eaurob-2024\UDI_PLV_2024.txt (1000, 41)
['distrcode', 'distrlib', 'distrsiret', 'cddept', 'inseecommune', 'nomcommune', 'cdreseau', 'nomreseau', 'codetypeinstallation', 'nomtypeinstallation', 'inae', 'cdpointsurv', 'nompointsurv', 'referenceprel', 'dateprel', 'heureprel', 'finaliteprel', 'conclusionprel', 'cdtypeeau', 'libtypeeau', 'plvconformitebacterio', 'plvconformitechimique', 'plvcomplet', 'plvcdorganismepayeur', 'typevisite', 'plvcondition', 'preleveur', 'distrsiren', 'plvconformiterefbacterio', 'plvconformiterefchimique', 'cddeptsandre', 'dtautorisation', 'champcaptant', 'codebrgm', 'codebss', 'dmajplv', 'dmodifplv', 'dmodifins', 'representativite', 'coord_x', 'coord_y']
                                                                          0  \
distrcode                                                            100547   
distrlib                                       SAUR FRANCE CHALON-SUR-SAONE   
distrsiret                                                 33937998404234.0 

In [18]:
import pandas as pd

codes = {"cddept": str, "inseecommune": str, "cdreseau": str, "referenceprel": str, "distrsiret": str}
plv = pd.read_csv("eaurob-2024\\UDI_PLV_2024.txt", sep=",", dtype=codes, low_memory=False)

print("Nombre de prélèvements :", len(plv))
print("Départements :", plv["cddept"].nunique(), sorted(plv["cddept"].unique())[:10])
for col in ["plvconformitebacterio", "plvconformitechimique",
            "plvconformiterefbacterio", "plvconformiterefchimique", "finaliteprel"]:
    print("\n", col)
    print(plv[col].value_counts(dropna=False))
print("\n", plv[["coord_x", "coord_y"]].describe())

res_cols = pd.read_csv("eaurob-2024\\UDI_RES_2024.txt", sep=",", nrows=5).columns.tolist()
print("\nColonnes de UDI_RES :", res_cols)

Nombre de prélèvements : 246631
Départements : 101 ['001', '002', '003', '004', '005', '006', '007', '008', '009', '010']

 plvconformitebacterio
plvconformitebacterio
C      217250
S       25292
N        4085
NaN         4
Name: count, dtype: int64

 plvconformitechimique
plvconformitechimique
C      240317
N        5916
D         380
S          14
NaN         4
Name: count, dtype: int64

 plvconformiterefbacterio
plvconformiterefbacterio
C      209983
S       25292
N       11352
NaN         4
Name: count, dtype: int64

 plvconformiterefchimique
plvconformiterefchimique
C      210057
N       36556
S          14
NaN         4
Name: count, dtype: int64

 finaliteprel
finaliteprel
CS    216384
CP     13376
S1     11872
S3      3206
CV      1015
CD       583
S7        54
S6        52
S5        42
S8        36
S4         6
S2         5
Name: count, dtype: int64

             coord_x       coord_y
count  1.482990e+05  1.483000e+05
mean   6.831569e+05  6.260524e+06
std    4.706349e+05  1.965

nettoyage csv


In [19]:
import pandas as pd

# 1. On lit seulement les 5 colonnes utiles, en gardant les codes en texte
colonnes = ["dateprel", "cddept", "nomcommune", "plvconformitebacterio", "plvconformitechimique"]
plv = pd.read_csv("eaurob-2024\\UDI_PLV_2024.txt", sep=",", usecols=colonnes, dtype=str)

# 2. On transforme la date en vraie date, et on ajoute le mois
plv["dateprel"] = pd.to_datetime(plv["dateprel"])
plv["mois"] = plv["dateprel"].dt.month

# 3. On remet les codes département au format habituel : "001" -> "01", "02A" -> "2A"
plv["dept"] = plv["cddept"].str.replace(r"^0", "", regex=True)

# 4. On regarde le résultat
print(plv.head())
print(plv["dept"].unique()[:15])

  cddept                  nomcommune   dateprel plvconformitebacterio  \
0    001  ABERGEMENT-CLEMENCIAT (L') 2024-02-09                     C   
1    001  ABERGEMENT-CLEMENCIAT (L') 2024-06-05                     C   
2    001  ABERGEMENT-CLEMENCIAT (L') 2024-10-04                     C   
3    001    ABERGEMENT-DE-VAREY (L') 2024-02-19                     C   
4    001    ABERGEMENT-DE-VAREY (L') 2024-07-25                     C   

  plvconformitechimique  mois dept  
0                     C     2   01  
1                     C     6   01  
2                     C    10   01  
3                     C     2   01  
4                     C     7   01  
<ArrowStringArray>
['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12', '13',
 '14', '15']
Length: 15, dtype: str


calculer les pourcentages de non-conformité

In [20]:
# 1. Pour chaque type de contrôle : évalué ? non conforme ?
plv["bacterio_evalue"] = plv["plvconformitebacterio"].isin(["C", "N"])
plv["bacterio_nc"] = plv["plvconformitebacterio"] == "N"
plv["chimique_evalue"] = plv["plvconformitechimique"].isin(["C", "N", "D"])
plv["chimique_nc"] = plv["plvconformitechimique"] == "N"

# 2. Taux national
taux_bact = plv["bacterio_nc"].sum() / plv["bacterio_evalue"].sum() * 100
taux_chim = plv["chimique_nc"].sum() / plv["chimique_evalue"].sum() * 100
print(f"France : {taux_bact:.1f} % non conformes en bactériologie, {taux_chim:.1f} % en chimie")

# 3. Taux par département
par_dept = plv.groupby("dept").agg(
    nb_prelevements=("dateprel", "size"),
    bact_evalues=("bacterio_evalue", "sum"),
    bact_nc=("bacterio_nc", "sum"),
    chim_evalues=("chimique_evalue", "sum"),
    chim_nc=("chimique_nc", "sum"),
)
par_dept["taux_bact"] = par_dept["bact_nc"] / par_dept["bact_evalues"] * 100
par_dept["taux_chim"] = par_dept["chim_nc"] / par_dept["chim_evalues"] * 100

print("\nLes 10 départements avec le plus de non-conformités bactériologiques :")
print(par_dept.sort_values("taux_bact", ascending=False).head(10).round(1))

# 4. Taux par mois (pour voir s'il y a un effet saison)
par_mois = plv.groupby("mois")[["bacterio_nc", "bacterio_evalue"]].sum()
par_mois["taux_bact"] = par_mois["bacterio_nc"] / par_mois["bacterio_evalue"] * 100
print("\nPar mois :")
print(par_mois["taux_bact"].round(2))

# 5. Vérification : tous les codes département
print("\nCodes département :", sorted(plv["dept"].unique()))

France : 1.8 % non conformes en bactériologie, 2.4 % en chimie

Les 10 départements avec le plus de non-conformités bactériologiques :
      nb_prelevements  bact_evalues  bact_nc  chim_evalues  chim_nc  \
dept                                                                  
48               2239          2212      316          2239       41   
2B               1556          1555      178          1555       27   
2A               1132          1121       99          1132       13   
05               2457          2208      194          2457        3   
09               2377          2194      179          2376       28   
07               3786          3297      260          3786       41   
26               2703          2630      189          2703       44   
973               552           549       36           549        4   
04               2408          2110      126          2408        8   
55               2447          2153      118          2447       66   

      taux_b

Ce qu’on découvre

Au niveau national, l’eau est très majoritairement conforme : seulement 1,8 % de prélèvements non conformes en bactériologie et 2,4 % en chimie.

Mais les problèmes bactériologiques sont très concentrés géographiquement. Regarde le haut du classement :

Dépt	Nom	Non conformes (bactério)
48	Lozère	14,3 %, soit 8 fois la moyenne nationale
2B	Haute-Corse	11,4 %
2A	Corse-du-Sud	8,8 %
05	Hautes-Alpes	8,8 %
09	Ariège	8,2 %
07	Ardèche	7,9 %

Ce sont presque tous des départements de montagne et ruraux. Une explication possible : ces territoires ont beaucoup de petits réseaux d’eau isolés, avec des moyens de traitement plus limités. Attention, c’est une hypothèse : les données ne le prouvent pas encore. On pourra la tester plus tard, par exemple en comptant le nombre de réseaux par département.

sauvegarder les résultats

In [21]:
import os
os.makedirs("data_processed", exist_ok=True)

# Les prélèvements nettoyés (pour les filtres de l'app)
plv.to_parquet("data_processed/prelevements_2024.parquet", index=False)

# Les tableaux agrégés (pour la carte et le graphique par mois)
par_dept.reset_index().to_parquet("data_processed/par_departement_2024.parquet", index=False)
par_mois.reset_index().to_parquet("data_processed/par_mois_2024.parquet", index=False)

print("Fichiers sauvegardés :", os.listdir("data_processed"))

# Le tableau par mois en entier (il était coupé dans ta sortie)
print(par_mois.round(2).to_string())

Fichiers sauvegardés : ['par_departement_2024.parquet', 'par_mois_2024.parquet', 'prelevements_2024.parquet']
      bacterio_nc  bacterio_evalue  taux_bact
mois                                         
1             169            17940       0.94
2             129            17329       0.74
3             143            17029       0.84
4             148            16572       0.89
5             212            16411       1.29
6             340            18079       1.88
7             449            19287       2.33
8             503            17171       2.93
9             538            17935       3.00
10            755            19477       3.88
11            420            20756       2.02
12            279            23349       1.19
